# Power BI Base Table Transformation

Este notebook implementa a lógica de transformação em Python para gerar uma tabela limpa que pode ser consumida pelo Power BI com atualização diária.

- Mantém todas as colunas da fonte original `a.*`.
- Aplica as transformações de joins e regras de autonomia definidas nas consultas do Power Query.
- Exporta o resultado para um arquivo local `parquet`, que pode ser apontado pelo Power BI.

In [18]:
import warnings
import numpy as np
import pandas as pd
import pyodbc
from pathlib import Path

OUTPUT_PATH = Path('output')
OUTPUT_PATH.mkdir(exist_ok=True)
DATA_INICIAL = '2026-01-01'

# Ajuste a string abaixo se o DSN exigir parâmetros adicionais.
# Para autenticação Kerberos, normalmente basta usar o DSN configurado.
KERBEROS_CONN_STR = 'DSN=DSN_HIVE_64;Authentication=Kerberos;'
# Se o driver exigir Trusted Connection no Windows, use:
# KERBEROS_CONN_STR = 'DSN=DSN_HIVE_64;Trusted_Connection=Yes;'


def get_connection() -> pyodbc.Connection:
    return pyodbc.connect(KERBEROS_CONN_STR, autocommit=True)


def query_to_df(query: str) -> pd.DataFrame:
    with get_connection() as conn:
        with warnings.catch_warnings():
            warnings.filterwarnings(
                'ignore',
                message='.*SQLAlchemy connectable.*',
                category=UserWarning,
                module='pandas.io.sql',
            )
            return pd.read_sql_query(query, conn)


## 1. RPS Parcial (vw_siplan_acao_2025_26)

Tabela de referência carregada primeiro: fornece atributos fixos por `atividade_id`.

Usada nas etapas seguintes para:
- Ajuste da `autonomiaTemporal` em DatasTotais (servico/subatividade)
- Cálculo de `autonomiaCusto` em Contratos (servico/subatividade)
- Cálculo da `autonomia` final em Autonomias (tag + tem_passagem)

**Campos:**
- `servico`, `subatividade` — tipo de realização e modalidade da ação
- `tag` — `'Avaliação STS'` se a atividade tem essa tag, senão `null`
- `tem_passagem` — `'Sim'` se há solicitação de passagem aérea vinculada

In [ ]:
sql_rps_parcial = '''
SELECT
    a.atividade_id,
    a.servico,
    a.subatividade,
    t.tag_nome                                                        AS tag,
    CASE WHEN p.atividade_id IS NOT NULL THEN 'Sim' ELSE 'Não' END   AS tem_passagem
FROM stg_estatistico.vw_siplan_acao_2025_26 AS a
LEFT JOIN stg_estatistico.vw_siplan_tags AS t
       ON t.atividade_id = a.atividade_id
      AND t.tag_nome = 'Avaliação STS'
LEFT JOIN (
    SELECT DISTINCT atividade_id
    FROM stg_estatistico.siplan_solicitacao
    WHERE item_grupo LIKE '%Passagem%'
) p ON p.atividade_id = a.atividade_id
WHERE a.servico IS NOT NULL
'''

In [ ]:
rps_parcial_df = query_to_df(sql_rps_parcial)
rps_parcial_df.columns = [col.split('.')[-1] for col in rps_parcial_df.columns]
rps_parcial_df = (
    rps_parcial_df
    .assign(
        atividade_id = rps_parcial_df['atividade_id'].astype(str).str.strip(),
        servico      = rps_parcial_df['servico'].astype(str).str.strip(),
        subatividade = rps_parcial_df['subatividade'].astype(str).str.strip(),
        tag          = rps_parcial_df['tag'].fillna(''),
        tem_passagem = rps_parcial_df['tem_passagem'].fillna('Não'),
    )
    .drop_duplicates(subset=['atividade_id'])
)
rps_parcial_df.shape

## 2. DatasTotais (vw_siplan_datas_totais)

Tabela `vw_siplan_datas_totais`: consolida, por `atividade_id`, a primeira data/hora das sessões e flags de autonomia temporal.

**Transformações aplicadas:**
- `primeiradata` → `PrimeiraData` (só data), `PrimeiraHora` (só hora), `PrimeiraDataHora` (string combinada)
- `tempo_da_sessao` = `tempo_sessao` truncado a 1 casa decimal
- `30dias` = `'sim'` se `diascorridos > 30`, senão `'0'`
- `90dias`, `60horas` = `'sim'`/`'0'` (mapeamento dos valores binários originais)
- `ExtrapolaDataHora` = `'sim'` se `90dias = 'sim'` **ou** `60horas = 'sim'`
- `mes` = abreviação do mês de `PrimeiraData`
- `autonomiaTemporal` (bruta) = `'DIREG'` se 90 dias **ou** 60 horas; `'STS'` se 30 dias; `'UO'` caso contrário
- `autonomiaTemporal` (ajustada via RPS Parcial): a autonomia temporal só se aplica a serviços que admitem acúmulo de carga horária/dias. Para os demais, é sempre `'UO'`:
  - **Usa temporal:** `Curso`, `Ioga`, `Desenvolvimento*`; subatividade que começa com `Ações`
  - **Sempre UO:** todos os outros serviços

In [19]:
sql_datas_totais = f'''
SELECT *
FROM stg_estatistico.vw_siplan_datas_totais
WHERE primeiradata > '{DATA_INICIAL}'
'''

In [ ]:
MONTH_ABBR = {1:'jan',2:'fev',3:'mar',4:'abr',5:'mai',6:'jun',
              7:'jul',8:'ago',9:'set',10:'out',11:'nov',12:'dez'}

def transform_datas_totais(df: pd.DataFrame, rps_df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Remove prefixo de nome de coluna que o driver ODBC pode incluir
    df.columns = [col.split('.')[-1] for col in df.columns]

    # Normaliza atividade_id para string (driver pode retornar float64)
    df['atividade_id'] = df['atividade_id'].astype(str).str.strip()

    # --- data/hora da primeira sessão ---
    df['primeiradata'] = pd.to_datetime(df['primeiradata'], errors='coerce')
    df['PrimeiraData']     = df['primeiradata'].dt.date
    df['PrimeiraHora']     = df['primeiradata'].dt.time
    df['PrimeiraDataHora'] = df['primeiradata'].dt.strftime('%Y-%m-%d %H:%M:%S')
    df['mes'] = df['primeiradata'].dt.month.map(MONTH_ABBR)

    # --- tempo da sessão (truncado a 1 decimal) ---
    df['tempo_da_sessao'] = (
        np.floor(pd.to_numeric(df['tempo_sessao'], errors='coerce').fillna(0) * 10) / 10
    )

    # --- flags de prazo (binário → 'sim'/'0') ---
    df['30dias'] = np.where(
        pd.to_numeric(df['diascorridos'], errors='coerce') > 30, 'sim', '0'
    )
    bool_map = {1: 'sim', '1': 'sim', 0: '0', '0': '0'}
    df['90dias']  = pd.to_numeric(df['90dias'],  errors='coerce').map(bool_map).fillna('0')
    df['60horas'] = pd.to_numeric(df['60horas'], errors='coerce').map(bool_map).fillna('0')
    df['ExtrapolaDataHora'] = np.where(
        (df['90dias'] == 'sim') | (df['60horas'] == 'sim'), 'sim', '0'
    )

    # --- autonomia temporal bruta ---
    # DIREG: ultrapassa 90 dias corridos OU 60 horas totais
    # STS  : ultrapassa 30 dias corridos
    # UO   : demais casos
    conditions = [
        (df['90dias'] == 'sim') | (df['60horas'] == 'sim'),
        df['30dias'] == 'sim',
    ]
    df['autonomiaTemporal'] = np.select(conditions, ['DIREG', 'STS'], default='UO')

    # --- ajuste por serviço (via RPS Parcial) ---
    # A autonomia temporal só faz sentido para serviços com acúmulo de carga/dias.
    # Para os demais, a UO é sempre o nível correto independente das datas.
    df = df.merge(rps_df[['atividade_id', 'servico', 'subatividade']], on='atividade_id', how='left')
    df['servico']      = df['servico'].fillna('')
    df['subatividade'] = df['subatividade'].fillna('')

    usa_temporal = (
        df['servico'].isin({'Curso', 'Ioga'}) |
        df['servico'].str.startswith('Desenvolvimento') |
        df['subatividade'].str.startswith('Ações')
    )
    df['autonomiaTemporal'] = np.where(usa_temporal, df['autonomiaTemporal'], 'UO')

    # servico/subatividade já estarão na tabela base; remove daqui para evitar duplicatas no merge
    df = df.drop(columns=['servico', 'subatividade'])

    return df


datas_totais_df = transform_datas_totais(query_to_df(sql_datas_totais), rps_parcial_df)
datas_totais_df.shape

## 3. Contratos (vw_siplan_contratos_calculados)

Tabela de solicitações de contratação vinculadas a cada `atividade_id`, com custo e público planejados.

**Transformações aplicadas:**
- `acima15mil`, `acima20mil`, `acima100mil` → `'sim'`/`'0'`
- `publico_sessao` (renomeado de `publico`): público por sessão conforme cadastro
- `publico` (recalculado): público total da ação
  - Se `publico_sessao > 4000`: valor já é total
  - Se `tipo_per_capita = 0`: total = `sessoes × publico_sessao`
  - Caso contrário: valor já é total
- `grupo`: remove sufixo `[...]` inserido pela view
- Merge com `rps_parcial_df` → traz `servico` e `subatividade`
- `autonomiaCusto`: nível de autonomia baseado no custo total de contratos
  - `DIREG` → acima de R$ 100 mil
  - `STS-20` → R$ 20–100 mil (serviços padrão)
  - `STS` → R$ 20–100 mil (serviços/subatividades com limiar especial de 20k)
  - `STS-15` → R$ 15–20 mil (serviços padrão)
  - `UO` → abaixo do limiar aplicável
  - Limiar especial de 20k: `subatividade` em {Ações formativas, Ações mediadas, Passeios, Viagens} ou `servico` em {Debate, Seminário, Visita Mediada}
- `por_hora_valido`: `por_hora` somente para serviços/subatividades onde faz sentido (Curso, Oficina, Vivência, Seminário, Mediação, Visita Mediada, Intervenção urbana; Multipráticas recreativas, Passeios, Viagens, Colônias recreativas)
- Deduplicação por `solicitacao_id`

In [21]:
sql_contracts = '''
SELECT
    a.atividade_id,
    a.solicitacao_id,
    a.grupo,
    a.item,
    a.complemento,
    a.custo,
    a.sessoes,
    a.horas,
    a.publico,
    a.capacidade,
    a.estimativa,
    a.por_sessao,
    a.por_hora,
    a.per_capita,
    a.tipo_per_capita,
    CASE WHEN COALESCE(a.custo_contratos_total, 0) > 15000  THEN 1 ELSE 0 END AS acima15mil,
    CASE WHEN COALESCE(a.custo_contratos_total, 0) > 20000  THEN 1 ELSE 0 END AS acima20mil,
    CASE WHEN COALESCE(a.custo_contratos_total, 0) > 100000 THEN 1 ELSE 0 END AS acima100mil,
    a.custo_total,
    a.custo_contratos_total,
    a.n_contratos,
    a.n_solic,
    d.primeiraData
FROM stg_estatistico.vw_siplan_contratos_calculados AS a
JOIN (
    -- filtra apenas atividades com pelo menos uma sessão em 2026
    SELECT
        atividade_id,
        MIN(datainicio) AS primeiraData
    FROM stg_estatistico.siplan_data
    GROUP BY atividade_id
    HAVING SUM(CASE WHEN YEAR(datainicio) = 2026 THEN 1 ELSE 0 END) > 0
) d ON a.atividade_id = d.atividade_id
'''

In [ ]:
SERVICOS_LIMIAR_20K  = {'Debate', 'Seminário', 'Visita Mediada'}
SUBATIV_LIMIAR_20K   = {'Ações formativas', 'Ações mediadas', 'Passeios', 'Viagens'}
SERVICOS_POR_HORA    = {'Curso', 'Oficina', 'Vivência', 'Seminário', 'Mediação', 'Visita Mediada', 'Intervenção urbana'}
SUBATIV_POR_HORA     = {'Multipráticas recreativas', 'Passeios', 'Viagens', 'Colônias recreativas'}

def transform_contracts(df: pd.DataFrame, rps_df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Remove prefixo de nome de coluna que o driver ODBC pode incluir
    df.columns = [col.split('.')[-1] for col in df.columns]

    # Normaliza atividade_id para string (driver pode retornar float64)
    df['atividade_id'] = df['atividade_id'].astype(str).str.strip()

    # --- flags de custo (binário → 'sim'/'0') ---
    bool_map = {1: 'sim', '1': 'sim', 0: '0', '0': '0'}
    for col in ['acima15mil', 'acima20mil', 'acima100mil']:
        df[col] = pd.to_numeric(df[col], errors='coerce').map(bool_map).fillna('0')

    # --- público total ---
    # A view retorna publico por sessão; o total depende do tipo de per capita:
    #   tipo_per_capita = 0 → público se repete a cada sessão → total = sessoes × publico
    #   tipo_per_capita ≠ 0 → público já é total (ou acima de 4000, claramente total)
    df = df.rename(columns={'publico': 'publico_sessao'})
    pub = pd.to_numeric(df['publico_sessao'],  errors='coerce').fillna(0)
    ses = pd.to_numeric(df['sessoes'],         errors='coerce').fillna(0)
    tpc = pd.to_numeric(df['tipo_per_capita'], errors='coerce').fillna(0)
    df['publico'] = np.where(pub > 4000, pub, np.where(tpc == 0, ses * pub, pub)).astype(int)

    # --- limpeza do grupo (remove sufixo "[...]" inserido pela view) ---
    df['grupo'] = df['grupo'].fillna('').astype(str).str.split('[').str[0].str.strip()

    # --- merge com RPS Parcial → servico e subatividade ---
    df = df.merge(rps_df[['atividade_id', 'servico', 'subatividade']], on='atividade_id', how='left')
    df['servico']      = df['servico'].fillna('')
    df['subatividade'] = df['subatividade'].fillna('')

    # --- autonomia de custo ---
    # Alguns serviços/subatividades usam limiar de R$ 20k (não R$ 15k) como base
    limiar_20k = (
        df['servico'].isin(SERVICOS_LIMIAR_20K) |
        df['subatividade'].isin(SUBATIV_LIMIAR_20K)
    )
    acima15  = df['acima15mil']  == 'sim'
    acima20  = df['acima20mil']  == 'sim'
    acima100 = df['acima100mil'] == 'sim'

    conditions = [
        acima100,                    # > 100k → DIREG sempre
        acima20 &  limiar_20k,       # 20k–100k, limiar especial → STS
        acima20 & ~limiar_20k,       # 20k–100k, padrão          → STS-20
        acima15 & ~limiar_20k,       # 15k–20k,  padrão          → STS-15
        # acima15 & limiar_20k → UO (abaixo do limiar especial de 20k)
    ]
    df['autonomiaCusto'] = np.select(conditions, ['DIREG', 'STS', 'STS-20', 'STS-15'], default='UO')

    # --- por_hora válido apenas para serviços que usam custo por hora ---
    usa_por_hora = (
        df['servico'].isin(SERVICOS_POR_HORA) |
        df['subatividade'].isin(SUBATIV_POR_HORA)
    )
    df['por_hora_valido'] = np.where(usa_por_hora, df['por_hora'], np.nan)

    # --- uma linha por solicitação ---
    df = df.drop_duplicates(subset=['solicitacao_id'])

    return df


contracts_df = transform_contracts(query_to_df(sql_contracts), rps_parcial_df)
contracts_df.shape

## 4. Autonomias

Combina as três tabelas anteriores para calcular a `autonomia` final de cada atividade.

**Fontes e o que contribuem:**
| Fonte | Campo | Regra |
|---|---|---|
| DatasTotais | `autonomiaTemporal` | DIREG (90d/60h), STS (30d), UO — ajustado por serviço |
| Contratos | `autonomiaCusto` | DIREG (>100k), STS-20 / STS (20k–100k), STS-15 (15k–20k), UO |
| RPS Parcial | `tag` | STS se `'Avaliação STS'` |
| RPS Parcial | `tem_passagem` | STS se `'Sim'` |

**Hierarquia:** `DIREG > STS > UO` — prevalece sempre o maior nível entre todas as fontes.

`autonomiaCusto` preserva o qualificador (`STS-15`, `STS-20`) para rastreabilidade, mas é tratado como nível STS na hierarquia.

In [ ]:
# Hierarquia: DIREG (3) > STS (2) > UO (1)
# Cada fonte de autonomia é convertida para nível numérico;
# o nível máximo determina a autonomia final.

NIVEL = {'DIREG': 3, 'STS': 2, 'STS-20': 2, 'STS-15': 2, 'UO': 1}

def autonomia_nivel(serie: pd.Series) -> pd.Series:
    """Converte valores de autonomia para nível numérico (desconhecido → 1)."""
    return serie.map(NIVEL).fillna(1).astype(int)

NIVEL_INV = {3: 'DIREG', 2: 'STS', 1: 'UO'}

def build_autonomias(rps_df, datas_df, contracts_df) -> pd.DataFrame:
    # Base: uma linha por atividade com serviço cadastrado
    df = rps_df[['atividade_id', 'servico', 'subatividade', 'tag', 'tem_passagem']].copy()

    # --- autonomia temporal (já ajustada por serviço em DatasTotais) ---
    df = df.merge(
        datas_df[['atividade_id', 'autonomiaTemporal']],
        on='atividade_id', how='left'
    )
    df['autonomiaTemporal'] = df['autonomiaTemporal'].fillna('UO')

    # --- autonomia de custo (uma linha por atividade) ---
    custo_por_ativ = (
        contracts_df[['atividade_id', 'autonomiaCusto']]
        .drop_duplicates(subset=['atividade_id'])
    )
    df = df.merge(custo_por_ativ, on='atividade_id', how='left')
    df['autonomiaCusto'] = df['autonomiaCusto'].fillna('UO')

    # --- níveis por fonte ---
    nivel_temporal  = autonomia_nivel(df['autonomiaTemporal'])
    nivel_custo     = autonomia_nivel(df['autonomiaCusto'])
    nivel_tag       = np.where(df['tag'] == 'Avaliação STS', 2, 1)
    nivel_passagem  = np.where(df['tem_passagem'] == 'Sim',  2, 1)

    # --- autonomia final: prevalece o maior nível (DIREG > STS > UO) ---
    nivel_final = pd.concat(
        [nivel_temporal, nivel_custo,
         pd.Series(nivel_tag, index=df.index),
         pd.Series(nivel_passagem, index=df.index)],
        axis=1
    ).max(axis=1)

    df['autonomia'] = nivel_final.map(NIVEL_INV)

    return df


autonomias_df = build_autonomias(rps_parcial_df, datas_totais_df, contracts_df)
autonomias_df['autonomia'].value_counts()